In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVR


In [2]:
path_folder = "Datasets/"
path_claim = "ClaimDetails_for_distribution.xlsx"

In [3]:
ds_claim = pd.read_excel(
    path_folder+path_claim,
    sheet_name=0
)

In [4]:
ds_claim = ds_claim.drop(columns=[
    "Claim Number",
    "Weather Indicator",
    "CAT Code",
    "Loss Date",
    "NOL Date",
    "Claimant Number",
    "Weather",
    "Loss Type",
    "Accident City",
    "Accident Zip"
])

In [5]:
# GLOBAL EXPERIMENT METRICS TABLE

metrics_table = pd.DataFrame(columns=[
    "Experiment",
    "Model",
    "TextStrategy",
    "Accuracy",
    "MAE",
    "QWK"
])

# FUNCTION TO ADD RESULTS

def add_experiment_result(
        metrics_table,
        experiment_name,
        model,
        text_strategy,
        accuracy,
        mae,
        qwk):

    new_row = pd.DataFrame([{
        "Experiment": experiment_name,
        "Model": model,
        "TextStrategy": text_strategy,
        "Accuracy": accuracy,
        "MAE": mae,
        "QWK": qwk
    }])

    metrics_table = pd.concat([metrics_table, new_row], ignore_index=True)

    return metrics_table

In [6]:
# MODEL L

import re
import numpy as np
import pandas as pd
from itertools import product

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, mean_absolute_error, cohen_kappa_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import LinearSVR

# 1. LOAD DATA
ds_claim = pd.read_excel(path_folder + path_claim, sheet_name=0).copy()


# 2. CONFIG
TARGET_COL = "CAT Severity Code"
RAW_TEXT_COL = "Cause Of Injury Text"

BASE_CATEGORICAL_COLS = [
    "Division",
    "Weather Text",
    "Peril Description"
]

NUMERIC_COLS = []

N_SPLITS = 5
RANDOM_STATE = 42

TFIDF_PARAMS = {
    "analyzer": "word",
    "ngram_range": (1, 4),
    "max_features": None,
    "min_df": 2,
    "sublinear_tf": True,
}

BASE_REGRESSOR = LinearSVR(
    C=1.0,
    epsilon=0.0,
    max_iter=10000,
    random_state=RANDOM_STATE
)

EXPERIMENT_NAME_BASE = "model_L_base"
EXPERIMENT_NAME_THRESH = "model_L_tresh"

# 3. BASIC CLEANING

df = ds_claim.copy()

df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = df[df[TARGET_COL].isin([1, 2, 3, 4, 5])].copy()
df[TARGET_COL] = df[TARGET_COL].astype(int)

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].astype(str).str.strip()
        df.loc[df[col].isin(["nan", "None", ""]), col] = np.nan

for col in BASE_CATEGORICAL_COLS:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        df.loc[df[col].isin(["nan", "None", ""]), col] = np.nan

for col in NUMERIC_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df[RAW_TEXT_COL] = df[RAW_TEXT_COL].fillna("").astype(str).str.strip()

# 4. RULE-BASED TAG EXTRACTION

def normalize_text(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"[^a-z0-9\s\/\-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def has_any(text: str, patterns) -> bool:
    return any(p in text for p in patterns)

def extract_event_family(text: str) -> str:
    if has_any(text, ["tornado", "hurricane", "storm", "wind", "gust", "shingle blown", "tree fell", "tree fall"]):
        return "wind"
    if has_any(text, ["hail", "ice dam", "ice "]):
        return "hail_ice"
    if has_any(text, ["water", "flood", "seepage", "leak", "overflow", "backup", "back-up", "sump", "sewer"]):
        return "water"
    if has_any(text, ["fire", "smoke", "burn", "char"]):
        return "fire"
    if has_any(text, ["theft", "stolen", "burglary", "robbery"]):
        return "theft"
    if has_any(text, ["collision", "crash", "struck", "hit", "rear end", "rear-end", "vehicle"]):
        return "collision_vehicle"
    if has_any(text, ["fall", "fell", "slip", "trip"]):
        return "fall_injury"
    if has_any(text, ["lightning"]):
        return "lightning"
    return "other"

def extract_damage_object(text: str) -> str:
    if has_any(text, ["roof", "shingle", "flashing", "attic"]):
        return "roof"
    if has_any(text, ["basement", "crawlspace", "crawl space", "foundation"]):
        return "basement_foundation"
    if has_any(text, ["window", "glass", "door", "garage door"]):
        return "window_door"
    if has_any(text, ["siding", "gutter", "fence", "exterior", "deck"]):
        return "exterior_structure"
    if has_any(text, ["ceiling", "wall", "floor", "drywall", "carpet", "kitchen", "bathroom", "interior"]):
        return "interior"
    if has_any(text, ["vehicle", "car", "truck", "auto"]):
        return "vehicle"
    if has_any(text, ["body", "injury", "claimant", "person", "medical"]):
        return "bodily_injury"
    return "other"

def extract_mechanism(text: str) -> str:
    if has_any(text, ["leak", "leaking", "roof leak", "flashing leak"]):
        return "leak"
    if has_any(text, ["seepage", "seep"]):
        return "seepage"
    if has_any(text, ["overflow", "backup", "back-up", "sump", "sewer"]):
        return "backup_overflow"
    if has_any(text, ["fell", "fall", "tree fell", "collapse", "collapsed"]):
        return "collapse_fall"
    if has_any(text, ["impact", "struck", "hit", "blown into"]):
        return "impact"
    if has_any(text, ["crack", "broken", "broke", "damage to"]):
        return "breakage"
    return "unknown"

def extract_catastrophe_flag(text: str, weather_text: str, peril_desc: str) -> str:
    combo = f"{text} {weather_text} {peril_desc}".lower()
    if has_any(combo, ["tornado", "hurricane", "tropical storm", "cat", "catastrophe"]):
        return "yes"
    return "no"

def extract_injury_flag(text: str, division: str) -> str:
    combo = f"{text} {division}".lower()
    if has_any(combo, ["injury", "injured", "claimant", "medical", "bodily", "slip", "trip", "fall"]):
        return "yes"
    if "pi" in combo or "personal" in combo:
        return "yes"
    return "no"

def extract_severity_cue(text: str) -> str:
    if has_any(text, ["total loss", "destroyed", "major", "severe", "extensive", "multiple", "entire", "collapsed"]):
        return "severe"
    if has_any(text, ["moderate", "significant", "substantial"]):
        return "moderate"
    if has_any(text, ["minor", "small", "slight", "limited"]):
        return "minor"
    return "unknown"

def extract_weather_specific(text: str, weather_text: str, peril_desc: str) -> str:
    combo = f"{text} {weather_text} {peril_desc}".lower()
    if "tornado" in combo:
        return "tornado"
    if "hurricane" in combo:
        return "hurricane"
    if "tropical storm" in combo:
        return "tropical_storm"
    if "hail" in combo:
        return "hail"
    if "wind" in combo or "storm" in combo:
        return "wind_storm"
    if "flood" in combo:
        return "flood"
    return "none"

def extract_location_context(text: str) -> str:
    if has_any(text, ["basement", "crawlspace", "crawl space"]):
        return "basement"
    if has_any(text, ["roof", "attic", "ceiling"]):
        return "roof_upper"
    if has_any(text, ["interior", "inside", "kitchen", "bathroom", "bedroom", "living room", "wall", "floor"]):
        return "inside"
    if has_any(text, ["outside", "exterior", "yard", "driveway", "fence", "siding"]):
        return "outside"
    return "unknown"

def build_semantic_tags(dataframe: pd.DataFrame) -> pd.DataFrame:
    temp = dataframe.copy()

    norm_text = temp[RAW_TEXT_COL].fillna("").astype(str).map(normalize_text)
    weather_text = temp["Weather Text"].fillna("").astype(str).map(normalize_text) if "Weather Text" in temp.columns else ""
    peril_desc = temp["Peril Description"].fillna("").astype(str).map(normalize_text) if "Peril Description" in temp.columns else ""
    division = temp["Division"].fillna("").astype(str).map(normalize_text) if "Division" in temp.columns else ""

    temp["tag_event_family"] = norm_text.map(extract_event_family)
    temp["tag_damage_object"] = norm_text.map(extract_damage_object)
    temp["tag_mechanism"] = norm_text.map(extract_mechanism)
    temp["tag_catastrophe_flag"] = [
        extract_catastrophe_flag(t, w, p) for t, w, p in zip(norm_text, weather_text, peril_desc)
    ]
    temp["tag_injury_flag"] = [
        extract_injury_flag(t, d) for t, d in zip(norm_text, division)
    ]
    temp["tag_severity_cue"] = norm_text.map(extract_severity_cue)
    temp["tag_weather_specific"] = [
        extract_weather_specific(t, w, p) for t, w, p in zip(norm_text, weather_text, peril_desc)
    ]
    temp["tag_location_context"] = norm_text.map(extract_location_context)

    return temp

df = build_semantic_tags(df)

SEMANTIC_TAG_COLS = [
    "tag_event_family",
    "tag_damage_object",
    "tag_mechanism",
    "tag_catastrophe_flag",
    "tag_injury_flag",
    "tag_severity_cue",
    "tag_weather_specific",
    "tag_location_context",
]

# 5. FEATURES

FEATURE_COLS = [RAW_TEXT_COL] + BASE_CATEGORICAL_COLS + SEMANTIC_TAG_COLS + NUMERIC_COLS
df_model = df[FEATURE_COLS + [TARGET_COL]].copy()

# 6. PREPROCESSOR

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

transformers = [
    ("text", TfidfVectorizer(**TFIDF_PARAMS), RAW_TEXT_COL),
    ("base_cat", categorical_transformer, BASE_CATEGORICAL_COLS),
    ("semantic_tags", categorical_transformer, SEMANTIC_TAG_COLS),
]

if len(NUMERIC_COLS) > 0:
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    transformers.append(("num", numeric_transformer, NUMERIC_COLS))

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop"
)

# 7. HELPERS

def clip_predictions(pred_continuous, lo=1.0, hi=5.0):
    return np.clip(pred_continuous, lo, hi)

def round_predictions(pred_continuous):
    return np.rint(clip_predictions(pred_continuous)).astype(int)

def apply_thresholds(pred_continuous, thresholds):
    pred = clip_predictions(pred_continuous)
    t1, t2, t3, t4 = thresholds
    return np.where(
        pred < t1, 1,
        np.where(pred < t2, 2,
        np.where(pred < t3, 3,
        np.where(pred < t4, 4, 5)))
    ).astype(int)

def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
        "qwk": cohen_kappa_score(y_true, y_pred, weights="quadratic"),
    }

def optimize_thresholds_grid(y_true, pred_continuous):
    best = {"qwk": -np.inf, "thresholds": None, "metrics": None}

    grid_t1 = np.arange(1.2, 2.21, 0.1)
    grid_t2 = np.arange(2.0, 3.21, 0.1)
    grid_t3 = np.arange(2.8, 4.21, 0.1)
    grid_t4 = np.arange(3.6, 4.81, 0.1)

    for t1, t2, t3, t4 in product(grid_t1, grid_t2, grid_t3, grid_t4):
        if not (t1 < t2 < t3 < t4):
            continue

        y_pred = apply_thresholds(pred_continuous, (t1, t2, t3, t4))
        m = compute_metrics(y_true, y_pred)

        if m["qwk"] > best["qwk"]:
            best = {
                "qwk": m["qwk"],
                "thresholds": (float(t1), float(t2), float(t3), float(t4)),
                "metrics": m
            }

    return best["thresholds"], best["metrics"]

# 8. CV TO GET OOF CONTINUOUS PREDICTIONS

def run_cv_oof_continuous(
    df_input,
    target_col,
    feature_cols,
    preprocessor,
    base_regressor,
    n_splits=5,
    random_state=42,
    experiment_name="experiment"
):
    X = df_input[feature_cols].copy()
    y = df_input[target_col].astype(int).values

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    oof_pred_cont = np.zeros(len(df_input), dtype=float)
    oof_pred_round = np.zeros(len(df_input), dtype=int)
    fold_rows = []

    for fold, (tr, va) in enumerate(skf.split(X, y), start=1):
        X_tr, X_va = X.iloc[tr].copy(), X.iloc[va].copy()
        y_tr, y_va = y[tr], y[va]

        X_tr_t = preprocessor.fit_transform(X_tr)
        X_va_t = preprocessor.transform(X_va)

        reg = clone(base_regressor)
        reg.fit(X_tr_t, y_tr)

        va_cont = clip_predictions(reg.predict(X_va_t))
        va_round = round_predictions(va_cont)

        oof_pred_cont[va] = va_cont
        oof_pred_round[va] = va_round

        m = compute_metrics(y_va, va_round)
        fold_rows.append({
            "experiment": experiment_name,
            "fold": fold,
            "n_train": len(tr),
            "n_valid": len(va),
            **m
        })

        print(
            f"{experiment_name} | Fold {fold}: "
            f"accuracy={m['accuracy']:.4f} | mae={m['mae']:.4f} | qwk={m['qwk']:.4f}"
        )

    oof_m = compute_metrics(y, oof_pred_round)
    fold_rows.append({
        "experiment": experiment_name,
        "fold": "OOF_round",
        "n_train": None,
        "n_valid": len(df_input),
        **oof_m
    })

    metrics_table_base = pd.DataFrame(fold_rows)

    oof_detail = df_input.copy()
    oof_detail["y_true"] = y
    oof_detail["y_pred_continuous"] = oof_pred_cont
    oof_detail["y_pred_round"] = oof_pred_round

    return metrics_table_base, oof_detail

# 9. RUN EXPERIMENT

metrics_table_base, oof_detail = run_cv_oof_continuous(
    df_input=df_model,
    target_col=TARGET_COL,
    feature_cols=FEATURE_COLS,
    preprocessor=preprocessor,
    base_regressor=BASE_REGRESSOR,
    n_splits=N_SPLITS,
    random_state=RANDOM_STATE,
    experiment_name=EXPERIMENT_NAME_BASE
)

print("\nBase metrics table (includes OOF_round row):")
print(metrics_table_base)

y_true = oof_detail["y_true"].values
pred_cont = oof_detail["y_pred_continuous"].values
pred_round = oof_detail["y_pred_round"].values

base_metrics = compute_metrics(y_true, pred_round)
base_cm = confusion_matrix(y_true, pred_round, labels=[1, 2, 3, 4, 5])

print("\nBaseline (naive rounding) OOF metrics:")
print(base_metrics)

print("\nBaseline confusion matrix (rows=true, cols=pred):")
print(pd.DataFrame(base_cm, index=[1,2,3,4,5], columns=[1,2,3,4,5]))

best_thresholds, best_thresh_metrics = optimize_thresholds_grid(y_true, pred_cont)
pred_thresh = apply_thresholds(pred_cont, best_thresholds)
thresh_cm = confusion_matrix(y_true, pred_thresh, labels=[1, 2, 3, 4, 5])

print("\nBest optimized thresholds found:")
print(best_thresholds)

print("\nOptimized-threshold OOF metrics:")
print(best_thresh_metrics)

print("\nOptimized-threshold confusion matrix (rows=true, cols=pred):")
print(pd.DataFrame(thresh_cm, index=[1,2,3,4,5], columns=[1,2,3,4,5]))

comparison_table = pd.DataFrame([
    {
        "experiment": EXPERIMENT_NAME_BASE,
        "mapping": "naive_round",
        "thresholds": (1.5, 2.5, 3.5, 4.5),
        **base_metrics
    },
    {
        "experiment": EXPERIMENT_NAME_THRESH,
        "mapping": "optimized_thresholds",
        "thresholds": best_thresholds,
        **best_thresh_metrics
    }
])

print("\nComparison table:")
print(comparison_table)

oof_detail["y_pred_threshopt"] = pred_thresh
oof_detail["thresholds_used"] = str(best_thresholds)

# 10. ADD TO YOUR METRICS TABLE

metrics_table = add_experiment_result(
    metrics_table,
    experiment_name="Model L",
    model="LinearSVR",
    text_strategy="Best TFIDF + semantic extracted tags + structured categoricals",
    accuracy=best_thresh_metrics["accuracy"],
    mae=best_thresh_metrics["mae"],
    qwk=best_thresh_metrics["qwk"]
)

print("\nExperiment leaderboard:")
print(metrics_table.sort_values("QWK", ascending=False))

# 11. OPTIONAL: INSPECT TAG DISTRIBUTIONS

for c in SEMANTIC_TAG_COLS:
    print(f"\n{c}")
    print(df[c].value_counts(dropna=False).head(10))
    

model_L_base | Fold 1: accuracy=0.4591 | mae=0.6792 | qwk=0.4973
model_L_base | Fold 2: accuracy=0.4747 | mae=0.6835 | qwk=0.4923
model_L_base | Fold 3: accuracy=0.4747 | mae=0.6962 | qwk=0.4894
model_L_base | Fold 4: accuracy=0.4367 | mae=0.7215 | qwk=0.4945
model_L_base | Fold 5: accuracy=0.4177 | mae=0.7532 | qwk=0.4586

Base metrics table (includes OOF_round row):
     experiment       fold  n_train  n_valid  accuracy       mae       qwk
0  model_L_base          1    632.0      159  0.459119  0.679245  0.497339
1  model_L_base          2    633.0      158  0.474684  0.683544  0.492292
2  model_L_base          3    633.0      158  0.474684  0.696203  0.489422
3  model_L_base          4    633.0      158  0.436709  0.721519  0.494546
4  model_L_base          5    633.0      158  0.417722  0.753165  0.458590
5  model_L_base  OOF_round      NaN      791  0.452592  0.706700  0.486410

Baseline (naive rounding) OOF metrics:
{'accuracy': 0.4525916561314791, 'mae': 0.706700379266751, 'qwk'

/tmp/ipykernel_1659668/798467192.py:32: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  metrics_table = pd.concat([metrics_table, new_row], ignore_index=True)


In [7]:
# FINAL INFERENCE PIPELINE - MODEL L

from sklearn.base import clone

# 1. Load full dataset
df_full = pd.read_excel(path_folder + path_claim, sheet_name=0).copy()

# 2. Apply same cleaning logic used before
df_full[TARGET_COL] = pd.to_numeric(df_full[TARGET_COL], errors="coerce")

for col in df_full.columns:
    if df_full[col].dtype == "object":
        df_full[col] = df_full[col].astype(str).str.strip()
        df_full.loc[df_full[col].isin(["nan", "None", ""]), col] = np.nan

for col in BASE_CATEGORICAL_COLS:
    if col in df_full.columns:
        df_full[col] = df_full[col].astype(str).str.strip()
        df_full.loc[df_full[col].isin(["nan", "None", ""]), col] = np.nan

df_full[RAW_TEXT_COL] = df_full[RAW_TEXT_COL].fillna("").astype(str).str.strip()

# 3. Recreate semantic tags on the full dataset
df_full = build_semantic_tags(df_full)

# 4. Split labeled vs missing severity
df_train = df_full[df_full[TARGET_COL].isin([1, 2, 3, 4, 5])].copy()
df_missing = df_full[df_full[TARGET_COL].isna()].copy()

X_train = df_train[FEATURE_COLS].copy()
y_train = df_train[TARGET_COL].astype(int).values

# 5. Fit preprocessing and model on ALL labeled data
X_train_t = preprocessor.fit_transform(X_train)

final_model = clone(BASE_REGRESSOR)
final_model.fit(X_train_t, y_train)

# 6. Predict missing CAT Severity Code using optimized thresholds
if len(df_missing) > 0:
    X_missing = df_missing[FEATURE_COLS].copy()
    X_missing_t = preprocessor.transform(X_missing)

    pred_cont = final_model.predict(X_missing_t)
    pred_final = apply_thresholds(pred_cont, best_thresholds)

    df_full.loc[df_full[TARGET_COL].isna(), TARGET_COL] = pred_final

    print(f"Filled {len(df_missing)} missing CAT Severity Code values")
else:
    print("No missing CAT Severity Code values to fill")

# 7. Final output
df_final = df_full.copy()

print("\nFinal dataset ready:")
print(df_final.head())

df_final.to_excel(f"{path_folder}claims_with_imputed_severity.xlsx", index=False)

Filled 274 missing CAT Severity Code values

Final dataset ready:
   Weather             Territory  Claim Number  CAT Code  CAT Severity Code  \
0  Weather      Alabama - Middle      25266097        82                3.0   
1  Weather      Alabama - Middle      86498344        82                5.0   
2  Weather      Alabama - Middle      11851998        82                5.0   
3  Weather       Alabama - North      15735237        82                2.0   
4  Weather  Metro Atlanta - West      50814161        82                3.0   

   Loss Date   NOL Date Peril Description  \
0 2023-12-09 2023-12-10              Wind   
1 2023-12-10 2023-12-10              Wind   
2 2023-12-10 2023-12-11              Wind   
3 2023-12-09 2023-12-21              Hail   
4 2023-12-10 2023-12-10              Wind   

                                Cause Of Injury Text Peril Group  ...  \
0  TORNADO CAME THROUGH AND TREE WENT THROUGH INS...        Wind  ...   
1  HEAVY STORMS WITH WIND, HAIL, AND TREES

In [8]:
df_final[df_final["Accident Zip"].isna()]

,Weather,Territory,Claim Number,CAT Code,CAT Severity Code,Loss Date,NOL Date,Peril Description,Cause Of Injury Text,Peril Group,...,6_cluster_cluster_id,6_cluster_distance_to_centroid_km,7_cluster_cluster_id,7_cluster_distance_to_centroid_km,8_cluster_cluster_id,8_cluster_distance_to_centroid_km,9_cluster_cluster_id,9_cluster_distance_to_centroid_km,10_cluster_cluster_id,10_cluster_distance_to_centroid_km
144,Weather,NaN,10318529,82,3.0,2023-12-09,2023-12-11,Wind,ROOF AND TWO OF THE BAY DOORS WERE DAMAGED DUE...,Wind,...,1,177.389085,1,177.389085,1,77.467248,1,77.467248,1,77.467248
767,Weather,NaN,40686860,82,3.0,2023-12-09,2023-12-13,Wind,WEDDING CANCELLED DUE TO TORNADO.,Wind,...,3,27.395343,3,27.395343,3,33.049635,3,33.049635,3,33.049635
1061,Weather,NaN,76778644,82,3.0,2023-12-09,2023-12-11,Wind,WHEEL PROS FACILITY IN TN HIT BY TORNADO,Wind,...,3,18.998116,3,18.998116,3,24.444078,3,24.444078,3,24.444078
